# 7장 — 생각의 길이: 추론 시점 컴퓨트

웹 챕터(7장)와 **같은 모형, 같은 수치**를 파이썬으로 직접 계산합니다.
같은 토큰 예산을 **길게 하나**(직렬) vs **짧게 여럿 + 다수결**(병렬)에 배분할 때,
문제 난이도와 예산에 따라 승자가 뒤집히는 것을 확인해요.

> 🤖 **AI 튜터와 함께 공부하세요** — 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch7')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 모형 — 생각의 길이에 따른 정답률 (교육용 toy)

풀이 길이 $L$(토큰)에 따른 정답률을 두 난이도로 모형화합니다 (웹 실험실과 동일):

- **쉬운 문제**: $a(L) = 0.95 - 0.45\,e^{-L/500}$ — 짧은 생각에 빨리 포화 (천장 95%)
- **어려운 문제**: $a(L) = 0.9 - 0.7\,e^{-L/4000}$ — 천천히 상승, 긴 생각이 필수 (천장 90%)

> ⚠️ 정직 고지: 실제 곡선은 모델·과업마다 다릅니다 — 이 모형은 감각을 잡기 위한 toy 이고,
> 실무 관행은 벤치마크 **실측**으로 배분을 정합니다.

예산 $B$의 두 전략 — ① 길게 하나: 정확도 $a(B)$ ② $k$개 다수결: 표 하나 $p=a(B/k)$ 에 5장의 이항 다수결.

In [ ]:
import math

def a_easy(L):
    return 0.95 - 0.45 * math.exp(-L / 500)

def a_hard(L):
    return 0.9 - 0.7 * math.exp(-L / 4000)

def maj(p, k):
    """k표(홀수) 다수결 정답률 — 5장의 이항 합."""
    return sum(math.comb(k, i) * p**i * (1 - p) ** (k - i) for i in range((k + 1) // 2, k + 1))

def strategies(a, B):
    """(길게 하나, 3표 다수결, 9표 다수결) 정답률."""
    return a(B), maj(a(B / 3), 3), maj(a(B / 9), 9)

BUDGETS = [1000, 2000, 4000, 8000, 16000, 32000, 64000]
for name, a in [('쉬운 문제', a_easy), ('어려운 문제', a_hard)]:
    print(f'== {name}')
    for B in BUDGETS:
        lo, m3, m9 = strategies(a, B)
        print(f'  예산 {B//1000:>2}k: 길게 하나 {lo*100:5.1f}%  셋이 투표 {m3*100:5.1f}%  아홉이 투표 {m9*100:5.1f}%')

# ── 웹 챕터 본문 수치 자가검증 ──
lo, m3, m9 = strategies(a_easy, 8000)
assert (round(lo*100,1), round(m3*100,1), round(m9*100,1)) == (95.0, 99.2, 99.7)  # 쉬움 8k
assert round(a_easy(8000/9)*100, 1) == 87.4  # 아홉이 투표의 표 하나
lo, m3, m9 = strategies(a_hard, 8000)
assert (round(lo*100,1), round(m3*100,1), round(m9*100,1)) == (80.5, 56.1, 15.4)  # 어려움 8k
assert round(a_hard(8000/9)*100, 1) == 33.9 and a_hard(8000/9) < 0.5  # 과반 미만 → 다수결 독
lo, m3, _ = strategies(a_hard, 32000)
assert (round(lo*100,1), round(m3*100,1)) == (90.0, 94.0)  # 어려움 32k — 역전
assert round(a_hard(32000/3)*100, 1) == 85.1
_, _, m9_1k = strategies(a_hard, 1000)
assert round(m9_1k*100, 1) == 2.9  # 소예산 아홉 투표의 참사
print('✅ 웹 실험실·본문 수치 자가검증 통과')

In [ ]:
# StepQuiz 재현 — 어려운 문제, 예산 10,000토큰, 셋이 투표
L = 10000 / 3
p = a_hard(L)
print(f'풀이 하나 길이 L = {L:.0f}토큰 → 모형 정답률 p = {p:.4f} (약 0.60)')
assert abs(p - 0.5958) < 5e-4

two = 3 * 0.6**2 * 0.4   # 정확히 2개 맞음
three = 0.6**3           # 3개 다 맞음
print(f'p=0.60 기준: 2개 맞음 {two:.3f} + 3개 다 맞음 {three:.3f} = 다수결 {two+three:.3f}')
assert abs(two - 0.432) < 1e-9 and abs(three - 0.216) < 1e-9 and abs(two + three - 0.648) < 1e-9

long10k = a_hard(10000)
print(f'같은 10,000토큰을 풀이 하나에: {long10k*100:.1f}% — 어려운 문제는 길게 하나의 승리')
assert round(long10k*100, 1) == 84.3
print('✅ StepQuiz 수치 자가검증 통과')

In [ ]:
# 교차점 — 예산 격자에서 '셋이 투표'가 '길게 하나'를 처음 이기는 지점
def first_win(a):
    for B in BUDGETS:
        lo, m3, _ = strategies(a, B)
        if m3 > lo:
            return B
    return None

easy_x, hard_x = first_win(a_easy), first_win(a_hard)
print(f'쉬운 문제: 예산 {easy_x//1000}k 부터 셋이 투표가 승리 (짧아도 각자 잘 풀어서)')
print(f'어려운 문제: 예산 {hard_x//1000}k 부터 — 쪼갠 풀이도 과반을 넘는 순간 다수결이 천장을 뚫는다')
assert easy_x == 4000 and hard_x == 32000
print('✅ 교차점 검증 통과 — 승자는 난이도와 예산이 정한다')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (name, a) in zip(axes, [('쉬운 문제', a_easy), ('어려운 문제', a_hard)]):
    for k, label, color in [(1, '길게 하나', '#6366f1'), (3, '셋이 투표', '#10b981'), (9, '아홉이 투표', '#f59e0b')]:
        ys = [(a(B) if k == 1 else maj(a(B / k), k)) * 100 for B in BUDGETS]
        ax.plot(BUDGETS, ys, marker='o', ms=4, color=color, label=label)
    ax.set_xscale('log')  # 10부 2장의 로그 축 — 수확 체감이 완만한 상승으로 보인다
    ax.axhline(50, ls='--', lw=1, color='#94a3b8')
    ax.set_title(f'{name} — 예산 배분별 정답률')
    ax.set_xlabel('토큰 예산 (로그 축)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('정답률 (%)')
axes[0].legend()
plt.tight_layout()
plt.show()

## 오늘의 발견

- **추론 시점 컴퓨트** = 답할 때 쓰는 계산 — 훈련(한 번 짓는 건축비)과 달리 **매 요청 지불하는 사용료**.
- 같은 예산의 두 배분: **직렬**(길게 하나) vs **병렬**(짧게 여럿 + 다수결) — 승자는 난이도·예산이 정한다:
  쉬운 문제는 4k부터 투표가 천장(95%)을 뚫고, 어려운 문제는 32k에서야 역전(쪼갠 풀이가 과반을 넘는 순간).
- 예산을 너무 쪼개면 표 하나가 과반 미만 → 다수결이 **독**(어려움 1k에서 아홉 투표 2.9%) — 5장의 역전 그대로.
- 수확 체감(로그 축)·과잉 생각·독립 가정 — 이론 곡선은 상한선, 실무는 실측(11장)으로 배분을 정한다.

다음 장(8장 — 갈림길 지도)은 병렬의 진화 — 아무거나 여럿이 아니라 **트리를 탐색**하는 이야기입니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**